# 00 序列建模与 Attention 前置背景

这一节是为了补上学习 Attention 和 QKV 前最容易缺的一块背景。

你不需要先完整学完 RNN，才能理解 Attention。

但你需要先知道几件事：

```text
什么是序列？
什么是 token？
为什么每个位置都可以表示成向量？
为什么一个位置要参考其他位置？
为什么早期序列模型会遇到长距离信息问题？
Attention 为什么换了一种思路？
```

这一节只补这些背景，不展开 RNN 代码，也不讲 Transformer 细节。

## 1. 什么是序列

序列就是一组有顺序的数据。

比如：

```text
一句话：我 今天 很 开心
一段时间序列：第 1 天销量、第 2 天销量、第 3 天销量
一首音乐：第 1 个音符、第 2 个音符、第 3 个音符
一张图片切成小块：第 1 个图像块、第 2 个图像块、第 3 个图像块
```

序列的重点不只是“有很多元素”，而是这些元素有位置关系。

同样几个词，顺序变了，意思可能就变了：

```text
我喜欢你
你喜欢我
```

词还是这些词，但位置不同，意思不同。

所以处理序列时，模型要关心两件事：

```text
每个位置本身是什么。
不同位置之间有什么关系。
```

## 2. 什么是 token

在文本任务里，经常把一句话拆成一个个基本单位。

这些基本单位可以先叫做 token。

为了入门理解，可以先把 token 理解成“词”或“字的一部分”。

例如一句话：

```text
小红 明天 要 考试
```

可以先粗略看成 4 个 token：

```text
token 1：小红
token 2：明天
token 3：要
token 4：考试
```

这时一句话就变成了一个长度为 4 的序列。

后面说的“第几个位置”，其实就是指第几个 token。

## 3. token 怎么进入神经网络

神经网络不能直接处理文字。

它处理的是数字。

所以每个 token 通常会先变成一个向量。

例如：

$$
\begin{aligned}
\text{小红} \rightarrow [0.2, -0.1, 0.7, \dots] \\
\text{明天} \rightarrow [0.0, 0.5, -0.3, \dots] \\
\text{考试} \rightarrow [0.8, 0.1, 0.4, \dots]
\end{aligned}
$$

这个向量可以理解成模型眼里的 token 表示。

它不是人手写出来的解释，而是模型学习出来的一组数字特征。

如果一句话有 $N$ 个 token，每个 token 用 $D$ 个数字表示，那么这句话就可以表示成：

$$
N\times D
$$

如果一次处理一批句子，batch size 是 $B$，就变成：

$$
B\times N\times D
$$

这就是后面 Attention 里常见的形状。

## 4. 为什么一个位置不能只看自己

看一句话：

```text
小明把复习资料交给小红，因为她明天要考试。
```

如果只看“她”这个 token 本身，我们很难知道它指谁。

要理解“她”，必须参考其他位置：

```text
小明
小红
复习资料
明天要考试
```

也就是说，一个位置的含义，经常需要结合上下文。

这里的上下文，就是这个位置周围或者远处的其他 token。

所以序列建模的一个核心问题是：

```text
当前位置如何利用其他位置的信息？
```

## 5. 早期思路：按顺序读过去

在 Attention 流行之前，处理序列的一类经典方法是 RNN 及其变体。

你现在不需要掌握 RNN 公式，只要先知道它的大致思路：

```text
从左到右，一个 token 一个 token 地读。
读到当前位置时，把前面读过的信息压缩到一个状态里。
```

比如读一句话：

```text
我 -> 今天 -> 很 -> 开心
```

RNN 会按顺序更新自己的内部状态。

可以先粗略理解成：

$$
\begin{aligned}
\text{当前状态} &= \text{当前 token 信息} + \text{前面已经读过的信息}
\end{aligned}
$$

这是一种很自然的序列处理方式。

因为人读句子时，也经常是从前往后读。

## 6. RNN 思路为什么会遇到困难

按顺序读有一个问题：信息要一站一站往后传。

如果两个词距离很近，信息传过去还比较容易。

如果距离很远，中间隔了很多 token，前面的信息就可能被冲淡。

可以把它想成传话：

```text
第 1 个人告诉第 2 个人
第 2 个人再告诉第 3 个人
第 3 个人再告诉第 4 个人
...
```

传得越远，信息越容易损失。

这就是为什么早期序列模型在长距离依赖上会比较吃力。

长距离依赖的意思是：

```text
当前位置需要参考很远之前或很远之后的信息。
```

注意：这不是说 RNN 完全不能处理长距离信息。

LSTM、GRU 等改进就是为了缓解这个问题。

这里只需要知道：按顺序传递信息，天然会让远距离关系变得不那么直接。

## 7. Attention 换了一种思路

Attention 的思路不是让信息一站一站往后传。

它更像是让当前位置直接去看所有位置。

还是那句话：

```text
小明把复习资料交给小红，因为她明天要考试。
```

当模型处理“她”这个位置时，它不必只依赖前面一步步传过来的压缩状态。

它可以直接和句子里的多个位置计算关系：

```text
她 和 小明 有多相关？
她 和 小红 有多相关？
她 和 复习资料 有多相关？
她 和 考试 有多相关？
```

然后把更相关的信息分配更大的权重。

这就是 Attention 对序列建模的核心变化：

```text
从按顺序传递信息，变成直接计算位置之间的关系。
```

## 8. 为什么会出现 $N \times N$

如果一句话有 $N$ 个 token，每个 token 都想看看其他 token。

那么就会出现很多“谁看谁”的关系。

举个长度为 4 的序列：

```text
token 1
token 2
token 3
token 4
```

token 1 可以看 4 个 token。

token 2 也可以看 4 个 token。

token 3 也可以看 4 个 token。

token 4 也可以看 4 个 token。

所以一共有：

$$
4\times4=16
$$

个关系分数。

如果是 $N$ 个 token，就是：

$$
N\times N
$$

这就是你在 QKV 里看到 $N\times N$ 分数表的原因。

它不是突然冒出来的矩阵，而是每个位置都去看每个位置。

## 9. 这和 QKV 有什么关系

现在回到 QKV。

每个 token 先有一个向量表示：

$$
\mathbf{x}_i
$$

Attention 会把它变成三种用途：

- $q_{i}$：这个位置想找什么。
- $k_{i}$：这个位置拿什么被别人匹配。
- $v_{i}$：这个位置真正提供什么内容。

于是“token i 看 token j”这件事，就可以说成：

用 $q_{i}$ 和 $k_{j}$ 计算相关性分数。

分数经过 Softmax 变成权重。

最后用这些权重去汇总各个位置的 $v_j$。

所以 QKV 不是凭空出现的。

它是在回答序列建模里的这个问题：

```text
每个位置应该怎样判断自己要参考哪些位置，并拿走哪些内容？
```

## 10. 你现在不需要完整补哪些内容

为了继续学 Attention，你现在不需要完整补完：

```text
RNN 的反向传播
LSTM 的门控公式
GRU 的门控公式
机器翻译 Encoder-Decoder 全流程
Transformer 的完整结构
```

这些以后可以学，但不是现在理解 QKV 的硬门槛。

你现在最需要先抓住的是：

```text
序列有多个位置。
每个位置有自己的向量表示。
一个位置的含义经常要参考其他位置。
Attention 让位置之间直接计算关系。
QKV 是 Attention 用来计算关系和汇总内容的分工。
```

## 11. 本节小结

这一节先记住几句话：

1. 序列是一组有顺序的数据，比如一句话里的多个 token。
2. 神经网络会先把每个 token 表示成一个向量。
3. 一句话可以看成 $N\times D$，一批句子可以看成 $B\times N\times D$。
4. 一个位置的含义往往需要参考其他位置。
5. RNN 的经典思路是按顺序读，把信息一步步往后传。
6. Attention 的思路是让位置之间直接计算关系。
7. $N\times N$ 来自“每个位置都看每个位置”。
8. QKV 是为了分别处理“提问、匹配、汇总内容”三件事。

## 12. 自测问题

1. 什么是序列？为什么顺序重要？
2. token 可以先粗略理解成什么？
3. 为什么神经网络不能直接处理文字？
4. 如果一句话有 $N$ 个 token，每个 token 用 $D$ 个数字表示，这句话的形状是什么？
5. 为什么一个位置不能只看自己？
6. RNN 处理序列的大致思路是什么？
7. 为什么按顺序传递信息会让长距离关系不够直接？
8. Attention 和 RNN 的主要思路区别是什么？
9. 为什么 $N$ 个位置两两计算关系会得到 $N\times N$？
10. QKV 分别对应“提问、匹配、汇总内容”中的哪一项？